In [ ]:
import pandas as pd
import numpy as np
import pandas_ta as ta
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import logging
import schedule
import MetaTrader5 as mt5
from datetime import datetime, timezone, timedelta
import time
import pytz
from threading import Event
import sys
import json
from config import ALLOWED_ELEMENT_TYPES,ICON_COLOR_MAP
from utils import reformat_scraped_data
from webdriver_manager.chrome import ChromeDriverManager
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# Initialize MetaTrader 5 connection and login
mt5.initialize()
username = int(os.environ['MT5_LOGIN'])
password = os.environ['MT5_PASSWORD']
server = 'Alpari-MT5-Demo'
mt5.login(username, password, server)

# Create a stop event
stop_event = Event()

# ANSI escape code for green text
GREEN = "\033[92m"
RESET = "\033[0m"

# Configure logging
logging.basicConfig(filename='trading_journal.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Create a console handler that writes to stdout
console_handler = logging.StreamHandler(stream=sys.stdout)
console_handler.setLevel(logging.INFO)

# Define a custom formatter that adds the green color
class CustomFormatter(logging.Formatter):
    def format(self, record):
        log_msg = super().format(record)
        return f"{GREEN}{log_msg}{RESET}"

console_handler.setFormatter(CustomFormatter('%(asctime)s - %(levelname)s - %(message)s'))

# Add the console handler to the root logger
logging.getLogger().addHandler(console_handler)

# Email configuration
SMTP_SERVER = 'smtp.zoho.com'
SMTP_PORT = 587
EMAIL_ADDRESS = os.environ['EMAIL_ADDRESS']
EMAIL_PASSWORD = os.environ['EMAIL_PASSWORD']

def send_email(subject, body):
    msg = MIMEMultipart()
    msg['From'] = EMAIL_ADDRESS
    msg['To'] = os.environ['ALERT_RECIPIENT']  # Intended recipient
    msg['Subject'] = subject

    msg.attach(MIMEText(body, 'plain'))

    try:
        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        text = msg.as_string()
        server.sendmail(EMAIL_ADDRESS, msg['To'], text)
        server.quit()
        logging.info('Notification Email sent successfully')
    except Exception as e:
        logging.error(f'Failed to send email: {e}')

def convert_time_to_mt5(df):
    # Define the time zones
    utc_plus1_tz = pytz.timezone('Etc/GMT-1')  # Assumed Forex Factory time zone
    mt5_tz = pytz.timezone('Etc/GMT-2')  # MT5 time zone

    # Convert the time column
    def convert_time(row):
        # Parse the time with a default date
        time_str = row['time']
        naive_time = datetime.strptime("2023-01-01 " + time_str, "%Y-%m-%d %I:%M%p")  # Convert to naive datetime
        
        # Localize to UTC+1
        localized_time = utc_plus1_tz.localize(naive_time)
        
        # Convert to MT5 time zone
        mt5_time = localized_time.astimezone(mt5_tz)
        
        return mt5_time.strftime("%I:%M %p")  # Return formatted string

    # Apply the conversion
    df['time'] = df.apply(convert_time, axis=1)
    return df
    
def news_fetch():
    try:
        from selenium import webdriver
        from selenium.webdriver.common.by import By
        driver = webdriver.Chrome()
    except:
        print ("AF: No Chrome webdriver installed")
        driver = webdriver.Chrome(ChromeDriverManager().install())

    driver.get("https://www.forexfactory.com/calendar")

    month =  datetime.now().strftime("%B")

    table = driver.find_element(By.CLASS_NAME, "calendar__table")

    data = []
    previous_row_count = 0
    # Scroll down to the end of the page
    while True:
        # Record the current scroll position
        before_scroll = driver.execute_script("return window.pageYOffset;")
        
        # Scroll down a fixed amount
        driver.execute_script("window.scrollTo(0, window.pageYOffset + 500);")
        
        # Wait for a short moment to allow content to load
        time.sleep(2)
        
        # Record the new scroll position
        after_scroll = driver.execute_script("return window.pageYOffset;")
        
        # If the scroll position hasn't changed, we've reached the end of the page
        if before_scroll == after_scroll:
            break

    # Now that we've scrolled to the end, collect the data
    for row in table.find_elements(By.TAG_NAME, "tr"):
        row_data = []
        for element in row.find_elements(By.TAG_NAME, "td"):
            class_name = element.get_attribute('class')
            if class_name in ALLOWED_ELEMENT_TYPES:
                if element.text:
                    row_data.append(element.text)
                elif "calendar__impact" in class_name:
                    impact_elements = element.find_elements(By.TAG_NAME, "span")
                    for impact in impact_elements:
                        impact_class = impact.get_attribute("class")
                        color = ICON_COLOR_MAP[impact_class]
                    if color:
                        row_data.append(color)
                    else:
                        row_data.append("impact")

        if len(row_data):
            data.append(row_data)

    reformat_scraped_data(data,month)
    ds = pd.read_csv(f'{month}_news.csv')
    ds = ds[ds['time'] != 'All Day']
    ds = ds[ds['currency'].isin(['USD', 'EUR', 'GBP'])]
    ds = ds[ds['impact'] == 'red']
    convert_time_to_mt5(ds)
    ds['date'] = ds['date'].astype(str).str.strip()
    current_year = datetime.now().year
    ds['date'] = ds['date'] + f' {current_year}'
    ds['date'] = pd.to_datetime(ds['date'], format='%b %d %Y', errors='coerce')
    filename = f"{month}_news.csv"
    ds.to_csv(filename)
    return ds

#get the red news
news_fetch()

def get_signal(ticker):
    if ticker == 'EURUSD_i':
        objects = joblib.load('U_E.joblib')
    elif ticker == 'GBPUSD_i':
        objects = joblib.load('U_G.joblib')
    elif ticker == 'XAGUSD_i':
        objects = joblib.load('U_XG.joblib')
    elif ticker == 'XAUUSD_i':
        objects = joblib.load('U_XU.joblib')

    interval = mt5.TIMEFRAME_M15
    rates = mt5.copy_rates_from_pos(ticker, interval, 1, 2100)
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.rename(columns={
        'time': 'Date',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close'
    }, inplace=True)
    df.set_index('Date', inplace=True)
    df = df.drop(columns=['tick_volume', 'real_volume', 'spread'])
    
    df['returns'] = df['Close'].pct_change()
    df['volatility'] = df['returns'].rolling(window=8).std()
    df['returns_div_volatility'] = df['returns'] / df['volatility'].replace(0, np.nan)
    df['returns_inv'] = 1 / df['returns'].replace(0, np.nan)
    df['volatility_inv'] = 1 / df['volatility'].replace(0, np.nan)
    df['atr'] = ta.atr(df['High'], df['Low'], df['Close'], length= 13)
    df[['lower', 'mid', 'upper', 'bandwidth', 'percent']] = ta.bbands(df['Close'], length=21, std=2)
    df[['stoch_k', 'stoch_d']] = ta.stoch(df['High'], df['Low'], df['Close'], 13, 3, 3)
    df['stoch'] = np.where(
        (df['stoch_k'] > df['stoch_d']) & (df['stoch_k'] <= 30), 1,
        np.where((df['stoch_k'] < df['stoch_d']) & (df['stoch_k'] >= 70), -1, 0)
    )
    df.dropna(inplace=True)

    try:
        scaler_hmm = objects['scaler_hmm']
        hmm_model = objects['hmm']
        feature_matrix = df[['returns', 'volatility']]
        feature_matrix_scaled = scaler_hmm.transform(feature_matrix)
        df['hmm_regime'] = hmm_model.predict(feature_matrix_scaled)
        probabilities = hmm_model.predict_proba(feature_matrix_scaled)
        df['hmm_prob_0'] = probabilities[:, 0]
        df['hmm_prob_1'] = probabilities[:, 1]
        df['hmm_prob_2'] = probabilities[:, 2]
        df['hmm_prob_3'] = probabilities[:, 3]
        df['hmm_prob_4'] = probabilities[:, 4]
    except Exception as e:
        print(f"hmm error: {e}")

    # Final cleanup
    df.dropna(inplace=True)

    
    scaler = objects['scaler']
    model = objects['clf']

    features = df.drop(columns=['Close', 'Open', 'High', 'Low', 'stoch_k', 'stoch_d', 'stoch', 'lower', 'mid', 'upper', 'bandwidth', 'percent', 'atr']).columns
    X = df[features]
    X = scaler.transform(X)
    X = pd.DataFrame(X, columns=features)
    df['pred'] = model.predict(X)
    df['pred'] = df['pred'].map({0: -1, 1: 1})
    probs = model.predict_proba(X)
    df['prob_class_0'] = probs[:, 0]  # Probability of class -1
    df['prob_class_1'] = probs[:, 1]   # Probability of class 1
    prob_1 = df['prob_class_1'].values[-1]
    prob_2 = df['prob_class_0'].values[-1]
    if prob_1 > prob_2:
        prob = prob_1
    else:
        prob = prob_2
    
    # kelly
    if ticker == 'EURUSD_i':
        win_loss_ratio = objects['E_wlr']
    elif ticker == 'GBPUSD_i':
        win_loss_ratio = objects['G_wlr']
    elif ticker == 'XAGUSD_i':
        win_loss_ratio = objects['X_wlr']
    elif ticker == 'XAUUSD_i':
        win_loss_ratio = objects['X_wlr']

    kelly = (prob - (1 - prob) / win_loss_ratio)
    prob = prob * 100
    prob = round(prob, 2)
    signal = df.pred.values[-1]
    signal_time = df.index[-1]
    signal_prev = df.pred.values[-2]
    upper = df.upper.values[-1]
    mid = df.mid.values[-1]
    lower = df.lower.values[-1]
    atr = df.atr.values[-1]
    stoch = df.stoch.values[-1]
    stoch_k = df.stoch_k.values[-1]
    stoch_d = df.stoch_d.values[-1]
    
    return signal, signal_time, signal_prev, upper, mid, lower, atr, prob, kelly, stoch, stoch_k, stoch_d

def calculate_lot_size(ticker, entry_price, entry_sl):

    # Get contract size and account balance
    symbol_info = mt5.symbol_info(ticker)
    contract_size = symbol_info.trade_contract_size
    account_balance = mt5.account_info().balance

    kelly = get_signal(ticker)[-4]
    risk = (((kelly * account_balance) * 0.55) * 0.1) / account_balance

    # Calculate the stop loss in pips
    sl_pip = round(abs(entry_price - entry_sl) * contract_size, 2)
    
    # Calculate the lot size
    lots = round((account_balance * risk) / sl_pip, 2)
    
    # Ensure the lot size is at least 0.01
    if lots < 0.01:
        lots = 0.01
    
    if lots > 49.99:
        lots = 49.99
    
    return lots

def get_open_position():
    positions = mt5.positions_get()
    if positions:
        return positions[0]  # Assuming only one position for simplicity
    return None

def close_position(position):
    ticket = position.ticket
    if position.type == mt5.ORDER_TYPE_BUY:
        close_action = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(position.symbol).bid
    else:
        close_action = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(position.symbol).ask
    
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": position.symbol,
        "volume": position.volume,
        "type": close_action,
        "position": ticket,
        "price": price,
        "comment": "close the position",
        "type_time": mt5.ORDER_TIME_GTC,
    }
    
    result = mt5.order_send(request)
    logging.info(f"Close order result: {result}")
    send_email('Position Closed', f'Position details: {result}')

def execute_trade(ticker, signal, qty, sl, tp):
    sl = round(sl, 5)
    tp = round(tp, 5)
    if signal == 1:
        order_type = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(ticker).ask
        action = "BUY"
    elif signal == -1:
        order_type = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(ticker).bid
        action = "SELL"
    else:
        logging.info("No action needed")
        return
    
    logging.info(f"Executing {action} order: {ticker}, Volume: {qty}, Price: {price}")
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": ticker,
        "volume": qty,
        "type": order_type,
        "price": price,
        "sl": sl,
        "tp": tp,
        "comment": "python open",
        "type_time": mt5.ORDER_TIME_GTC,
    }
    result = mt5.order_send(request)
    logging.info(f"Trade order result: {result}")
    send_email('Position Opened', f'Position details: {result}')


def get_mt5_time():
    tz_mt5 = pytz.timezone('Etc/GMT-2')  # Use the correct timezone
    now = datetime.now(tz_mt5)
    return now.strftime('%Y-%m-%d %H:%M:%S')
    

def get_next_bar_time(interval):
    now = get_mt5_time()
    now = datetime.strptime(now, '%Y-%m-%d %H:%M:%S')
    if interval == mt5.TIMEFRAME_M15:
        minutes_past = now.minute % 15
        minutes_to_next_bar = (15 - minutes_past) % 15
        if minutes_to_next_bar == 0:
            minutes_to_next_bar = 15
        
        # Calculate the exact next bar time
        next_bar_time = now.replace(second=0, microsecond=0) + timedelta(minutes=minutes_to_next_bar)
        return next_bar_time
    else:
        raise ValueError("Unsupported timeframe")

today_balance = mt5.account_info().balance
# Function to fetch and update today's balance
def update_today_balance():
    global today_balance  # Use the global variable to store the balance
    today_balance = mt5.account_info().balance
    logging.info(f'Today balance: {today_balance}')

def fetch_data(ticker):
    interval = mt5.TIMEFRAME_M15
    rates = mt5.copy_rates_from_pos(ticker, interval, 1, 99999)
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.rename(columns={
        'time': 'Date',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close'
    }, inplace=True)
    df.set_index('Date', inplace=True)
    df = df.drop(columns=['tick_volume', 'real_volume', 'spread'])

    # target calculation
    df['returns'] = df['Close'].pct_change()
    df['target'] = (df['returns'] > 0).astype(int)
    df['s_returns'] = df['returns']
    df['cumulative_buy_and_hold_returns'] = df["returns"].cumsum().apply(np.exp)

    # Shift price and returns to prevent lookahead bias
    df['Close'] = df['Close'].shift(1)
    df['Open'] = df['Open'].shift(1)
    df['High'] = df['High'].shift(1)
    df['Low'] = df['Low'].shift(1)
    df['returns'] = df['returns'].shift(1)
    df['volatility'] = df['returns'].rolling(window=8).std()
    df['returns_div_volatility'] = df['returns'] / df['volatility'].replace(0, np.nan)
    df['returns_inv'] = 1 / df['returns'].replace(0, np.nan)
    df['volatility_inv'] = 1 / df['volatility'].replace(0, np.nan)
    df[['stoch_k', 'stoch_d']] = ta.stoch(df['High'], df['Low'], df['Close'], 13, 3, 3)
    df['stoch'] = np.where(
        (df['stoch_k'] > df['stoch_d']) & (df['stoch_k'] <= 30), 1,
        np.where((df['stoch_k'] < df['stoch_d']) & (df['stoch_k'] >= 70), -1, 0)
    )
    df.dropna(inplace=True)

    #HMM
    feature_matrix = df[['returns', 'volatility']]
    scaler_hmm = StandardScaler()
    feature_matrix_scaled = scaler_hmm.fit_transform(feature_matrix)
    hmm_model = GaussianHMM(n_components = 5, covariance_type="full", n_iter = 100 , random_state=42)
    hmm_model.fit(feature_matrix_scaled)
    df['hmm_regime'] = hmm_model.predict(feature_matrix_scaled)
    probabilities = hmm_model.predict_proba(feature_matrix_scaled)
    df['hmm_prob_0'] = probabilities[:, 0]
    df['hmm_prob_1'] = probabilities[:, 1]
    df['hmm_prob_2'] = probabilities[:, 2]
    df['hmm_prob_3'] = probabilities[:, 3]
    df['hmm_prob_4'] = probabilities[:, 4]

    # Final clean up
    df.replace([np.inf, -np.inf], 0, inplace=True)
    df.dropna(inplace=True)

    # features
    features = df.drop(columns=['target', 's_returns', 'cumulative_buy_and_hold_returns', 'Close', 'Open', 'High', 'Low', 'stoch_k', 'stoch_d','stoch']).columns

    # Define features and target
    X = df[features]
    y = df["target"]

    # Apply StandardScaler
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Convert the scaled NumPy array back to a DataFrame
    X = pd.DataFrame(X, columns=features)

    # Split the dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf_clf = RandomForestClassifier(n_estimators=500,
                                    max_depth=5,
                                    class_weight = 'balanced',
                                    random_state=42)

    model = rf_clf

    # Train the model on the training set
    model.fit(X_train, y_train)

    # Predict and calculate accuracy for training and testing sets
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    accuracy_train = accuracy_score(y_train, y_pred_train) * 100
    accuracy_test = accuracy_score(y_test, y_pred_test) * 100

    # Predict on the entire dataset
    df['pred'] = model.predict(X)
    total_accuracy = accuracy_score(df.target, df.pred) * 100

    # Get predicted probabilities for each class
    probs = model.predict_proba(X)

    # Add these probabilities to the dataframe
    df['prob_class_-1'] = probs[:, 0]
    df['prob_class_1'] = probs[:, 1]
    df['prob'] = np.where(df['prob_class_1'] > df['prob_class_-1'], df['prob_class_1'], df['prob_class_-1'])

    # first filtration
    df_filtered = df[((df['stoch'] == 1) & (df['pred'] == 1)) | ((df['stoch'] == -1) & (df['pred'] == -1))].copy()
    df_filtered['pred'] = df_filtered['pred'].map({0: -1, 1: 1})
    df_filtered['strategy'] = df_filtered['pred'] * df_filtered['s_returns']
    df_filtered['loss'] = np.where(df_filtered['strategy'] < 0, abs(df_filtered['strategy']), np.nan)
    df_filtered['win'] = np.where(df_filtered['strategy'] > 0, df_filtered['strategy'], np.nan)
    win_loss_ratio = df_filtered['win'].mean() / df_filtered['loss'].mean()
    win_loss_ratio = np.where(np.isnan(win_loss_ratio) | (win_loss_ratio == 0), 1, win_loss_ratio)
    df_filtered['kelly'] = (df_filtered['prob'] - (1 - df_filtered['prob']) / win_loss_ratio)

    # second filtration
    df_filtered_2 = df_filtered[(df_filtered['kelly'] > 0)].copy()
    df_filtered_2['pred'] = df_filtered_2['pred'].map({-1: 0, 1: 1})
    accuracy = accuracy_score(df_filtered_2.target, df_filtered_2.pred) * 100
    df_filtered_2['pred'] = df_filtered_2['pred'].map({0: -1, 1: 1})
    df_filtered_2['strategy'] = df_filtered_2['pred'] * df_filtered_2['s_returns']
    df_filtered_2['cumulative_strategy_returns'] = df_filtered_2["strategy"].cumsum().apply(np.exp)
    df_filtered_2['loss'] = np.where(df_filtered_2['strategy'] < 0, abs(df_filtered_2['strategy']), np.nan)
    df_filtered_2['win'] = np.where(df_filtered_2['strategy'] > 0, df_filtered_2['strategy'], np.nan)
    win_loss_ratio = df_filtered_2['win'].mean() / df_filtered_2['loss'].mean()
    win_loss_ratio = np.where(np.isnan(win_loss_ratio) | (win_loss_ratio == 0), 1, win_loss_ratio)
    df_filtered['kelly'] = (df_filtered_2['prob'] - (1 - df_filtered_2['prob']) / win_loss_ratio)
    annualized_factor = 24960  # The number of 15-minute bars in a year
    annualized_sreturn = np.mean(df_filtered_2['strategy']) * annualized_factor
    risk_free_rate = 0  # Assuming risk-free rate is 0 for simplicity
    mean_return = np.mean(df_filtered_2['strategy']) * annualized_factor
    std_return = np.std(df_filtered_2['strategy']) * np.sqrt(annualized_factor)
    sharpe_ratio = (mean_return - risk_free_rate) / std_return
    def max_drawdown(strategy_returns):
        cumulative = np.cumsum(strategy_returns)
        drawdown = cumulative - np.maximum.accumulate(cumulative)
        return np.min(drawdown)
    max_dd = max_drawdown(df_filtered_2['strategy'])
    position_counts = len(df_filtered_2)
    cumulative_strategy_returns = df_filtered_2['cumulative_strategy_returns'].iloc[-1]

    return accuracy, win_loss_ratio, annualized_sreturn, sharpe_ratio, max_dd, model, hmm_model, scaler, scaler_hmm, position_counts, cumulative_strategy_returns

def retrain():
    XU_variables = joblib.load('U_XU.joblib')
    accuracy, win_loss_ratio, annualized_sreturn, sharpe_ratio, max_dd, model, hmm_model, scaler, scaler_hmm, position_counts, cumulative_strategy_returns = fetch_data('XAUUSD_i')
    if (accuracy >= XU_variables['X_accuracy_th'] and abs(max_dd) <= XU_variables['X_max_dd_th'] and position_counts >= XU_variables['X_position_count_th'] and cumulative_strategy_returns >= XU_variables['X_cumulative_th']):
        objects_to_save = {
            'clf': model,
            'hmm': hmm_model,
            'scaler': scaler,
            'scaler_hmm': scaler_hmm,
            'X_wlr': win_loss_ratio,
            'X_accuracy_th': accuracy,
            'X_sharp_ratio_th': sharpe_ratio,
            'X_max_dd_th': max_dd,
            'X_position_count_th': position_counts,
            'X_cumulative_th': cumulative_strategy_returns
            }
        joblib.dump(objects_to_save, 'U_XU.joblib')
        logging.info(f'XAUUSD models retrained and replaced.\nAcuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')
        send_email('XAUUSD models retrained and replaced', f'Acuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')
    else:
        logging.info(f'We checked XAUUSD retraining results and we are going to reuse previous model this month.\nAcuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')
        send_email('XAUUSD models retraining results', f'We checked XAUUSD retraining results and we are going to reuse previous model this month.\nAcuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')

    
    XG_variables = joblib.load('U_XG.joblib')
    accuracy, win_loss_ratio, annualized_sreturn, sharpe_ratio, max_dd, model, hmm_model, scaler, scaler_hmm, position_counts, cumulative_strategy_returns = fetch_data('XAGUSD_i')
    if (accuracy >= XG_variables['X_accuracy_th'] and abs(max_dd) <= XG_variables['X_max_dd_th'] and position_counts >= XG_variables['X_position_count_th'] and cumulative_strategy_returns >= XG_variables['X_cumulative_th']):
        objects_to_save = {
            'clf': model,
            'hmm': hmm_model,
            'scaler': scaler,
            'scaler_hmm': scaler_hmm,
            'X_wlr': win_loss_ratio,
            'X_accuracy_th': accuracy,
            'X_sharp_ratio_th': sharpe_ratio,
            'X_max_dd_th': max_dd,
            'X_position_count_th': position_counts,
            'X_cumulative_th': cumulative_strategy_returns
            }
        joblib.dump(objects_to_save, 'U_XG.joblib')
        logging.info(f'XAGUSD models retrained and replaced.\nAcuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')
        send_email('XAGUSD models retrained and replaced', f'Acuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')
    else:
        logging.info(f'We checked XAGUSD retraining results and we are going to reuse previous model this month.\nAcuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')
        send_email('XAGUSD models retraining results', f'We checked XAGUSD retraining results and we are going to reuse previous model this month.\nAcuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')


    E_variables = joblib.load('U_E.joblib')
    accuracy, win_loss_ratio, annualized_sreturn, sharpe_ratio, max_dd, model, hmm_model, scaler, scaler_hmm, position_counts, cumulative_strategy_returns = fetch_data('EURUSD_i')
    if (accuracy >= E_variables['E_accuracy_th'] and abs(max_dd) <= E_variables['E_max_dd_th'] and position_counts >= E_variables['E_position_count_th'] and cumulative_strategy_returns >= E_variables['E_cumulative_th']):
        objects_to_save = {
            'clf': model,
            'hmm': hmm_model,
            'scaler': scaler,
            'scaler_hmm': scaler_hmm,
            'E_wlr': win_loss_ratio,
            'E_accuracy_th': accuracy,
            'E_sharp_ratio_th': sharpe_ratio,
            'E_max_dd_th': max_dd,
            'E_position_count_th': position_counts,
            'E_cumulative_th': cumulative_strategy_returns
            }
        joblib.dump(objects_to_save, 'U_E.joblib')
        logging.info(f'EURUSD models retrained and replaced.\nAcuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')
        send_email('EURUSD models retrained and replaced', f'Acuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')
    
    else:
        logging.info(f'We checked EURUSD retraining results and we are going to reuse previous model this month.\nAcuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')
        send_email('EURUSD models retraining results', f'We checked EURUSD retraining results and we are going to reuse previous model this month.\nAcuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')


    G_variables = joblib.load('U_G.joblib')
    accuracy, win_loss_ratio, annualized_sreturn, sharpe_ratio, max_dd, model, hmm_model, scaler, scaler_hmm, position_counts, cumulative_strategy_returns = fetch_data('GBPUSD_i')
    if (accuracy >= G_variables['G_accuracy_th'] and abs(max_dd) <= G_variables['G_max_dd_th'] and position_counts >= G_variables['G_position_count_th'] and cumulative_strategy_returns >= G_variables['G_cumulative_th']):
        objects_to_save = {
            'clf': model,
            'hmm': hmm_model,
            'scaler': scaler,
            'scaler_hmm': scaler_hmm,
            'G_wlr': win_loss_ratio,
            'G_accuracy_th': accuracy,
            'G_sharp_ratio_th': sharpe_ratio,
            'G_max_dd_th': max_dd,
            'G_position_count_th': position_counts,
            'G_cumulative_th': cumulative_strategy_returns
            }
        joblib.dump(objects_to_save, 'U_G.joblib')
        logging.info(f'GBPUSD models retrained and replaced.\nAcuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')
        send_email('GBPUSD models retrained and replaced', f'Acuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')

    else:
        logging.info(f'We checked GBPUSD retraining results and we are going to reuse previous model this month.\nAcuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')
        send_email('GBPUSD models retraining results', f'We checked GBPUSD retraining results and we are going to reuse previous model this month.\nAcuuracy: {accuracy}\nShrp ratio: {sharpe_ratio}\nMaximum drawdown: {max_dd}\nPosition counts: {position_counts}\nCumulative strategy returns: {cumulative_strategy_returns}')

# Schedule the function to run every day at midnight in the specified timezone
def schedule_news():
    timezone = pytz.timezone('Etc/GMT-2')
    now = datetime.now(timezone)
    schedule_time = now.replace(hour=0, minute=7, second=0, microsecond=0)

    # If it's already past midnight, schedule for the next day
    if now > schedule_time:
        schedule_time += timedelta(days=1)

    schedule.every().day.at(schedule_time.strftime("%H:%M")).do(news_fetch)

def schedule_retrain():
    timezone = pytz.timezone('Etc/GMT-2')
    now = datetime.now(timezone)

    # Calculate the next 21st of the month at midnight
    if now.day == 21:
        schedule_time = now.replace(hour=0, minute=7, second=0, microsecond=0)
        # If today is already the 21st but time is past midnight, set for next month
        if now > schedule_time:
            next_month = now.month + 1 if now.month < 12 else 1
            next_year = now.year if next_month > 1 else now.year + 1
            schedule_time = datetime(next_year, next_month, 21, 0, 7, 0, tzinfo=timezone)
    else:
        # Set schedule_time to the next 21st of the month
        next_month = now.month + 1 if now.month < 12 else 1
        next_year = now.year if next_month > 1 else now.year + 1
        schedule_time = datetime(next_year, next_month, 21, 0, 7, 0, tzinfo=timezone)

    # Calculate time difference and schedule retrain function
    time_diff = (schedule_time - now).total_seconds()

    # Schedule the retrain to run at the calculated time
    schedule.every(time_diff).seconds.do(retrain)

def schedule_balance_update():
    timezone = pytz.timezone('Etc/GMT-2')
    now = datetime.now(timezone)
    schedule_time = now.replace(hour=0, minute=7, second=0, microsecond=0)

    # If it's already past midnight, schedule for the next day
    if now > schedule_time:
        schedule_time += timedelta(days=1)

    schedule.every().day.at(schedule_time.strftime("%H:%M")).do(update_today_balance)

def check_and_trade(stop_event, news_df):
    # Initial scheduling
    schedule_news()
    schedule_retrain()
    schedule_balance_update()
    
    while not stop_event.is_set():
        try:
            # Run scheduled tasks
            schedule.run_pending()
            
            # Print MT5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            logging.info(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)
            next_check_time = next_bar_time + timedelta(seconds=1)
            # logging.info(f"Next check time: {next_check_time}")

            # Filter news events for the current day
            current_date = MT5.strftime('%b %d')  # Format current date as "Sep 24"
            news_df['date'] = pd.to_datetime(news_df['date'], errors='coerce')
            news_df_today = news_df[news_df['date'].dt.strftime('%b %d') == current_date]

            #print(f"Current date: {current_date}")
            #logging.info(f"News events for today:\n{news_df_today[['date', 'time', 'currency','impact','event']]}")
            logging.info(f"--------------------------------------")

            # Fetch open position
            open_position = get_open_position()

            # Check if MT5 falls on a weekend (Saturday or Sunday)
            if MT5.weekday() >= 5 or (MT5.weekday() == 4 and MT5.time() >= datetime.strptime("21:21", "%H:%M").time()):
                if open_position:
                    close_position(open_position)
                    logging.info(f"Its almost weekend, position closed.")

                logging.info("It's the weekend and the market is closed.")
                send_email('Market closed', 'It’s the weekend, and the market is closed. we are continue trading on Monday.')

                # Loop until Monday
                while MT5.weekday() >= 5 or (MT5.weekday() == 4 and MT5.time() >= datetime.strptime("21:21", "%H:%M").time()):
                    time.sleep(333)  # Wait a bit before checking again
                    MT5 = get_mt5_time()
                    MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')  # Update MT5 to the current datetime

                # Resume trading on Monday
                logging.info("It's Monday, and we are continue trading...")
                send_email('Market Opened', 'It’s Monday, and the market has opened. Continue trading...')
                             
            # Wait until 1 seconds after the bar closes
            while MT5 < next_check_time:
                time.sleep(1)  # Sleep briefly to avoid busy waiting
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')

            # Check for upcoming news events
            for index, row in news_df_today.iterrows():
                news_time = datetime.strptime(row['time'], '%I:%M %p')  # Adjusted format to match '04:45 PM'
                news_time = news_time.replace(year=MT5.year, month=MT5.month, day=MT5.day)  # Ensure the correct date
                message_logged = False  # Flag to check if the message has already been logged

                while news_time - timedelta(minutes=29) <= MT5 < news_time + timedelta(minutes=15):
                    # Fetch open position
                    open_position = get_open_position()
                    
                    if open_position:
                        close_position(open_position)
                        logging.info("Closed open position due to upcoming news event.")
                    
                    if not message_logged:  # Log only if the message hasn't been logged yet
                        logging.info(f"News event at {news_time} for {row['currency']} and the news is {row['event']}. Halting trading.")
                        message_logged = True  # Set the flag to True after logging the message
                    
                    time.sleep(1)
                    MT5 = get_mt5_time()
                    MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            
            # Fetch open position
            open_position = get_open_position()

            if open_position:
                current_ticker = open_position.symbol
                current_signal, signal_time, signal_prev, upper, mid, lower, atr, prob, kelly, stoch, stoch_k, stoch_d = get_signal(current_ticker)
                if (stoch_k <= 55 and stoch_k > stoch_d and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (stoch_k >= 45 and stoch_k < stoch_d and open_position.type == mt5.ORDER_TYPE_BUY):
                    logging.info(f"Stoch crossed over, closing position.")
                    close_position(open_position)
                else:
                    logging.info(f"We already have an open position")
            
            open_position = get_open_position()
            if open_position is None:
                # Fetch current signal and its time
                current_signal, signal_time, signal_prev, upper, mid, lower, atr, prob, kelly, stoch, stoch_k, stoch_d = get_signal('EURUSD_i')
                ticker = 'EURUSD_i'
                logging.info(f"Results for {ticker}: current signal: {current_signal}, stoch: {stoch}, kelly: {kelly} ")

                # Check if initial signal meets criteria
                if (current_signal == 1 and stoch == 0) or (current_signal == -1 and stoch == 0) or kelly <= 0:
                    current_signal, signal_time, signal_prev, upper, mid, lower, atr, prob, kelly, stoch, stoch_k, stoch_d = get_signal('GBPUSD_i')
                    ticker = 'GBPUSD_i'
                    logging.info(f"Results for {ticker}: current signal: {current_signal}, stoch: {stoch}, kelly: {kelly} ")

                    if (current_signal == 1 and stoch == 0) or (current_signal == -1 and stoch == 0) or kelly <= 0:
                        current_signal, signal_time, signal_prev, upper, mid, lower, atr, prob, kelly, stoch, stoch_k, stoch_d = get_signal('XAGUSD_i')
                        ticker = 'XAGUSD_i'
                        logging.info(f"Results for {ticker}: current signal: {current_signal}, stoch: {stoch}, kelly: {kelly} ")

                        if (current_signal == 1 and stoch == 0) or (current_signal == -1 and stoch == 0) or kelly <= 0:
                            current_signal, signal_time, signal_prev, upper, mid, lower, atr, prob, kelly, stoch, stoch_k, stoch_d = get_signal('XAUUSD_i')
                            ticker = 'XAUUSD_i'
                            logging.info(f"Results for {ticker}: current signal: {current_signal}, stoch: {stoch}, kelly: {kelly} ")

                # Final check and logging
                if (current_signal == 1 and stoch == 1 and kelly > 0) or (current_signal == -1 and stoch == -1 and kelly > 0):
                    # Check if the signal time is 30 minutes behind current MT5 time
                    time_diff = MT5 - signal_time

                    # Get the deals history and check sl
                    from_date = MT5 - timedelta(days=1)  # Last day of deals
                    deals = mt5.history_deals_get(from_date, MT5)

                    # get current balance
                    current_balance = mt5.account_info().balance
                    daily_drawdown_balance_limit = today_balance - (today_balance * 0.02)

                    if time_diff > timedelta(minutes=30):
                        logging.info("The signal is old.")
                        if open_position:
                            close_position(open_position)
                        continue  # Skip further processing
                    
                    if deals:
                        # Define groups of tickers
                        ticker_group_a = ['EURUSD_i', 'GBPUSD_i']
                        ticker_group_b = ['XAGUSD_i', 'XAUUSD_i']

                        # Define SL time limit
                        time_sl_check = (MT5 - timedelta(hours=12)).replace(tzinfo=None)  # Make offset-naive

                        # Initialize a flag
                        skip_signal = False

                        # Iterate through odd-indexed deals
                        for i in range(-1, -len(deals) - 1, -1):
                            deal = deals[i]

                            # Convert deal time without adding timezone
                            deal_time_mt5 = datetime.fromtimestamp(deal.time).replace(tzinfo=None)  # Make offset-naive

                            # Retrieve deal details
                            deal_symbol = deal.symbol
                            deal_type = deal.type
                            deal_comment = deal.comment
                            comment_sl_check = 'sl' in deal_comment.lower()

                            # Check if the current ticker and deal belong to the same group
                            if (ticker in ticker_group_a and deal_symbol in ticker_group_a) or \
                            (ticker in ticker_group_b and deal_symbol in ticker_group_b):

                                # Check if the signal matches the last deal and hit SL within the time limit
                                if ((deal_type == 1 and current_signal == 1) or (deal_type == 0 and current_signal == -1)) and \
                                comment_sl_check and deal_time_mt5 >= time_sl_check:
                                    logging.info(f'The signal is {current_signal} in {ticker} which is the same exact position that hit SL in group {deal_symbol} and time is {deal_time_mt5}, skipping this signal.')
                                    skip_signal = True
                                    break

                        if skip_signal:
                            continue

                    if current_balance < daily_drawdown_balance_limit:
                        logging.info(f'We already hit the daily drawdown limit, there will be no position today.\nCurrent balance: {current_balance}\nToday balance: {today_balance}\nDaily drawdown balance limit{daily_drawdown_balance_limit}')
                        continue  # Skip further processing

                    logging.info(f"WE have new signal in {ticker}: pred is {current_signal} and stoch is {stoch}, Signal time: {signal_time}, Probability: {prob} %, Kelly: {kelly}")
                    if current_signal == 1 and stoch == 1 and kelly > 0:
                        entry = mt5.symbol_info_tick(ticker).ask
                        if entry < mid:
                            sl = lower - (1.3 * atr)
                            tp = mid
                        elif entry > mid:
                            sl = mid  - (1.3 * atr)
                            tp = upper
                        lot = calculate_lot_size(ticker, entry, sl)
                        execute_trade(ticker, current_signal, lot, sl, tp)
                    elif current_signal == -1 and stoch == -1 and kelly > 0:
                        entry = mt5.symbol_info_tick(ticker).bid
                        if entry < mid:
                            sl = mid  + (1.3 * atr)
                            tp = lower
                        elif entry > mid:
                            sl = upper + (1.3 * atr)
                            tp = mid
                        lot = calculate_lot_size(ticker, entry, sl)
                        execute_trade(ticker, current_signal, lot, sl, tp)
                else:
                    logging.info("WE don't have a new signal yet.")
        except Exception as e:
            logging.info(f"An error occurred: {e}")
            # raise
            send_email('An error occured', f'The error is {e}')
            time.sleep(60)  # Wait before retrying in case of error

In [ ]:
try:
    # Get the current month name
    current_month = datetime.now().strftime('%B')
    
    # Construct the file name
    file_name = f'{current_month}_news.csv'
    
    # Read the news CSV file into a DataFrame
    news_df = pd.read_csv(file_name)

    # Start the trading loop
    check_and_trade(stop_event, news_df)

except KeyboardInterrupt:
    # Handle manual interruption
    print("Interrupted by user")
finally:
    # Shutdown MetaTrader 5 connection when done
    mt5.shutdown()
    print("MetaTrader 5 connection closed")